# UABAF — Uncertainty-Aware Bias Auditing Framework
### User Guide & Demo Notebook

**Package:** `uabaf`  
**Author:** Balgah Sounders Junior  
**Institution:** University of Buea, Department of Computer Engineering  

---

## What is UABAF?

UABAF is a Python package for auditing fairness in machine learning models trained on **small datasets**.

Standard fairness tools report a single number:
```
Demographic Parity Difference: 0.12  ← FAIL
```

On small datasets, that number could shift significantly with a different random seed. UABAF adds a **BCa bootstrap confidence interval** and an **uncertainty-aware verdict**:
```
Demographic Parity Difference: 0.12  CI: [0.04, 0.21]  🔍 FAIL — Low Confidence
```

This tells you not just *what* the metric is, but *how much you can trust it*.

---

## Three-Stage Pipeline

| Stage | What it does |
|-------|-------------|
| **Stage 1** — Dataset Profiling | Checks subgroup size, class imbalance, proxy features, missing data |
| **Stage 2** — Metric Computation | Computes DPD, EOD, EOP, DI with BCa bootstrap CIs |
| **Stage 3** — Verdict Assignment | Assigns one of four uncertainty-aware verdicts per metric |

---

## Verdict Categories

| Verdict | Meaning |
|---------|--------|
| ✅ PASS — High Confidence | Metric within threshold, narrow CI — result is reliable |
| ⚠️ PASS — Low Confidence | Metric within threshold but CI is wide — collect more data |
| ❌ FAIL — High Confidence | Metric breaches threshold, narrow CI — bias finding is reliable |
| 🔍 FAIL — Low Confidence | Metric breaches threshold but CI is wide — result is inconclusive |

---
## 1. Installation

Install UABAF from PyPI. The `[compare]` extra adds AIF360 and Fairlearn for comparison experiments.

In [ ]:
# Core package
# pip install uabaf

# With AIF360 + Fairlearn comparison support
# pip install uabaf[compare]

# Verify installation
import uabaf
print(f"UABAF version: {uabaf.__version__}")

---
## 2. Quick Start — The Simplest Way to Use UABAF

If you just want a full audit in as few lines as possible, use `AuditReport`.
It runs all three stages automatically.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedShuffleSplit
from uabaf import AuditReport

# ── Example: synthetic dataset ────────────────────────────────────────────────
np.random.seed(42)
n         = 200
X         = np.random.randn(n, 4)
sensitive = np.tile([1, 0], n // 2)       # 1 = privileged, 0 = unprivileged
y         = (X[:, 0] + X[:, 1] > 0).astype(int)

# Introduce bias against unprivileged group
y[sensitive == 0] = np.where(
    np.random.rand((sensitive == 0).sum()) > 0.45,
    0, y[sensitive == 0]
)

# Train / test split
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
for train_idx, test_idx in sss.split(X, sensitive):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    s_train, s_test = sensitive[train_idx], sensitive[test_idx]

# Train a model
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

# ── Run the full UABAF audit ──────────────────────────────────────────────────
report = AuditReport(
    model        = clf,
    X_test       = X_test,
    y_test       = y_test,
    sensitive    = s_test,
    model_name   = 'Logistic Regression',
    dataset_name = 'Synthetic Data',
)

# Print the verdict table
report.summary()

In [ ]:
# Plot the BCa confidence interval chart
import matplotlib.pyplot as plt

fig = report.plot()
plt.show()

In [ ]:
# Export results to a DataFrame — useful for saving to CSV or further analysis
df_results = report.to_dataframe()
df_results

---
## 3. Using UABAF Stage by Stage

Advanced users can call each stage independently for more control.

### Stage 1 — Dataset Readiness Profiling

In [ ]:
import pandas as pd
from uabaf.profiler import profile_dataset

# Build a DataFrame from your data
df = pd.DataFrame(X_train, columns=['feature_1','feature_2','feature_3','feature_4'])
df['sensitive'] = s_train
df['label']     = y_train

# Run Stage 1
profile = profile_dataset(
    df           = df,
    features     = ['feature_1','feature_2','feature_3','feature_4'],
    sensitive    = 'sensitive',
    target       = 'label',
    dataset_name = 'My Dataset',
)

profile.print_report()

In [ ]:
# Access findings as a DataFrame
profile.to_dataframe()

### Stage 2 — Fairness Metrics with BCa Confidence Intervals

In [ ]:
from uabaf.bootstrap import bca_ci_all_metrics, bca_ci

y_pred = clf.predict(X_test)

# All four metrics in one call (recommended — most efficient)
ci_results = bca_ci_all_metrics(
    y_true       = y_test,
    y_pred       = y_pred,
    sensitive    = s_test,
    B            = 2000,       # bootstrap resamples
    alpha        = 0.05,       # significance level → 95% CI
    random_state = 42,
)

for metric, result in ci_results.items():
    print(f"{metric.upper()}: point={result['point']:.4f}  "
          f"CI=[{result['ci_lo']:.4f}, {result['ci_hi']:.4f}]  "
          f"width={result['width']:.4f}")

In [ ]:
# Single metric CI — useful when you only need one
dpd_result = bca_ci(
    y_true       = y_test,
    y_pred       = y_pred,
    sensitive    = s_test,
    metric_key   = 'dpd',
    B            = 2000,
    random_state = 42,
)
print(dpd_result)

### Stage 3 — Verdict Assignment

In [ ]:
from uabaf.verdict import assign_all_verdicts, assign_verdict

# All four verdicts from Stage 2 output
verdicts = assign_all_verdicts(ci_results)

for metric, v in verdicts.items():
    print(f"{v['label']:<35} {v['verdict']}")

In [ ]:
# Single verdict — useful for custom workflows
v = assign_verdict('dpd', point=0.12, ci_lo=0.04, ci_hi=0.21)
print(v['verdict'])
print(f"Pass/Fail : {v['pass_fail']}")
print(f"Confidence: {v['confidence']}")

### Using Custom Thresholds

For high-stakes applications (e.g. medical, legal) you may want stricter thresholds.

In [ ]:
from uabaf.verdict import assign_all_verdicts

# Stricter thresholds — half the default tolerance
strict_thresholds = {
    'dpd': ('diff',  0.05),
    'eod': ('diff',  0.05),
    'eop': ('diff',  0.05),
    'di' : ('ratio', (0.90, 1.10)),
}

strict_verdicts = assign_all_verdicts(
    ci_results,
    thresholds = strict_thresholds
)

print("Strict threshold verdicts:")
for metric, v in strict_verdicts.items():
    print(f"  {v['label']:<35} {v['verdict']}")

---
## 4. Individual Metric Functions

You can also call the four fairness metric functions directly.

In [ ]:
from uabaf.metrics import (
    demographic_parity_diff,
    equalized_odds_diff,
    equal_opportunity_diff,
    disparate_impact_ratio,
    fairness_metrics,
)

# All four at once
metrics = fairness_metrics(y_test, y_pred, s_test)
print("All metrics:")
for k, v in metrics.items():
    print(f"  {k.upper()}: {v:.4f}")

print()

# Or individually
print(f"DPD: {demographic_parity_diff(y_test, y_pred, s_test):.4f}")
print(f"EOD: {equalized_odds_diff(y_test, y_pred, s_test):.4f}")
print(f"EOP: {equal_opportunity_diff(y_test, y_pred, s_test):.4f}")
print(f"DI : {disparate_impact_ratio(y_test, y_pred, s_test):.4f}")

---
## 5. Visualisation

UABAF provides two plot types.

In [ ]:
from uabaf.visualise import plot_ci_intervals, plot_verdict_summary
from sklearn.ensemble import RandomForestClassifier

# ── Plot 1: CI interval chart for one model ───────────────────────────────────
fig = plot_ci_intervals(
    verdicts,
    title = 'Synthetic Data — Logistic Regression'
)
plt.show()

In [ ]:
# ── Plot 2: Verdict summary grid comparing two models ─────────────────────────
# Train a second model for comparison
rf  = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_pred    = rf.predict(X_test)
rf_ci      = bca_ci_all_metrics(y_test, rf_pred, s_test, B=2000, random_state=42)
rf_verdicts = assign_all_verdicts(rf_ci)

summary_data = {
    'Logistic Regression': verdicts,
    'Random Forest'      : rf_verdicts,
}

fig2 = plot_verdict_summary(
    summary_data,
    title = 'Model Comparison — Synthetic Data'
)
plt.show()

---
## 6. Real Dataset Example — German Credit

Demonstrating UABAF on one of the three thesis datasets.
Sensitive attribute: `sex` (1=male, 0=female). Target: `credit_risk`.

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedShuffleSplit
from uabaf import AuditReport

# ── Load German Credit dataset ────────────────────────────────────────────────
url = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
       "statlog/german/german.data")
col_names = [
    'checking_account','duration','credit_history','purpose',
    'credit_amount','savings_account','employment','installment_rate',
    'personal_status_sex','other_debtors','residence_since',
    'property','age','other_installment','housing',
    'existing_credits','job','dependents','telephone',
    'foreign_worker','credit_risk'
]
df = pd.read_csv(url, sep=' ', header=None, names=col_names)

# Encode sensitive attribute and target
df['sex']         = df['personal_status_sex'].apply(
                        lambda x: 0 if x in ['A92','A95'] else 1)
df['credit_risk'] = df['credit_risk'].map({1: 1, 2: 0})

# One-hot encode categoricals
cat_cols = ['checking_account','credit_history','purpose','savings_account',
            'employment','personal_status_sex','other_debtors','property',
            'other_installment','housing','job','telephone','foreign_worker']
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

sensitive = 'sex'
target    = 'credit_risk'
features  = [c for c in df.columns if c not in [sensitive, target]]

# ── Train / test split ────────────────────────────────────────────────────────
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
for train_idx, test_idx in sss.split(df[features], df[sensitive]):
    X_tr = df.iloc[train_idx][features].values
    X_te = df.iloc[test_idx][features].values
    y_tr = df.iloc[train_idx][target].values
    y_te = df.iloc[test_idx][target].values
    s_te = df.iloc[test_idx][sensitive].values

# ── Train and audit ───────────────────────────────────────────────────────────
clf = LogisticRegression(max_iter=1000, random_state=42).fit(X_tr, y_tr)

report = AuditReport(
    model        = clf,
    X_test       = X_te,
    y_test       = y_te,
    sensitive    = s_te,
    model_name   = 'Logistic Regression',
    dataset_name = 'German Credit',
)

report.summary(show_stage1=False)
fig = report.plot()
plt.show()

---
## 7. Saving Results

Export audit results to CSV or save the plot as an image.

In [ ]:
# Save verdict table to CSV
df_results = report.to_dataframe()
df_results.to_csv('german_credit_audit.csv', index=False)
print("Results saved to german_credit_audit.csv")

# Save CI plot as image
fig = report.plot()
fig.savefig('german_credit_audit.png', dpi=150, bbox_inches='tight')
print("Plot saved to german_credit_audit.png")
plt.show()

---
## 8. API Reference

### `AuditReport` — main class
```python
AuditReport(model, X_test, y_test, sensitive,
            B=2000, alpha=0.05, random_state=42,
            model_name='Model', dataset_name='Dataset',
            df=None, features=None, target=None)
```
| Method | Returns | Description |
|--------|---------|-------------|
| `.summary()` | None | Prints Stage 1 + Stage 3 verdict table |
| `.plot()` | Figure | BCa CI interval chart |
| `.to_dataframe()` | DataFrame | Results as a pandas DataFrame |

---

### Individual functions
```python
from uabaf.profiler  import profile_dataset       # Stage 1
from uabaf.bootstrap import bca_ci_all_metrics    # Stage 2 — all metrics
from uabaf.bootstrap import bca_ci                # Stage 2 — single metric
from uabaf.verdict   import assign_all_verdicts   # Stage 3 — all metrics
from uabaf.verdict   import assign_verdict        # Stage 3 — single metric
from uabaf.visualise import plot_ci_intervals     # CI bar chart
from uabaf.visualise import plot_verdict_summary  # Multi-model grid
from uabaf.metrics   import fairness_metrics      # Raw metric values
```

---

### Fairness metric definitions

| Key | Metric | Formula | Fair Range |
|-----|--------|---------|------------|
| `dpd` | Demographic Parity Difference | P(ŷ=1\|S=1) − P(ŷ=1\|S=0) | \|dpd\| ≤ 0.10 |
| `eod` | Equalized Odds Difference | max(\|TPR diff\|, \|FPR diff\|) | ≤ 0.10 |
| `eop` | Equal Opportunity Difference | \|TPR_priv − TPR_unpriv\| | ≤ 0.10 |
| `di`  | Disparate Impact Ratio | P(ŷ=1\|S=0) / P(ŷ=1\|S=1) | 0.80 – 1.20 |

---

## Links

- **PyPI:** https://pypi.org/project/uabaf
- **GitHub:** https://github.com/PtDabam/uabaf
- **Reference:** Efron & Tibshirani (1993) — *An Introduction to the Bootstrap*, Chapter 14